# Gap-filling head time series using Pastas

*Raoul Collenteur, Hydroconsult, 2026*

This notebook shows how to use the Pastas gapfilling module to gapfill a head time series using a simulation from a Pastas model. Rather than 'simply' imputing the simulation values in the gaps, the simulation is corrected using the measurements surrounding the gap and the noise model (if available). Making use of a hydrological model (here, a Pastas model), ensure that we take the hydrology causing the head fluctuations into account. This is different from interpolation, which ignores the fact that it did or did not rain, for example. As usual, we start by importing the necessary Python packages.

The development of this module was sponsored by Waterboard Vallei & Veluwe.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

import pastas as ps

ps.show_versions()

## Example data

We look at the data from Rhode Island (USA), where the head measurements have an approximately daily frequency. We'll make some synthetic gaps that we can fill using a Pastas Model and the gapfill module. We only use every 14th measurement, and create a gap for the entire year with the lowest measured head (2007). 

In [ ]:
data = ps.load_dataset("collenteur_2019")
head = data["head"].squeeze().loc[:"2012"]

head_gap = head.iloc[::14].copy()
head_gap.loc[:"2003-12"] = pd.NA
head_gap.loc["2007-06":"2007-12"] = pd.NA
head_gap.loc["2010-01":"2010-06"] = pd.NA

head_gap = head_gap.dropna()
prec = data["rain"].asfreq("D").fillna(0) * 0.3048 * 1e3
evap = data["evap"].asfreq("D").interpolate() * 0.3048 * 1e3

## Make a Pastas Model

We'll make a simple Pastas Model based on precipitation and potential evaporation to simulate the head at a daily time step. The simulation time step from the Pastas Model is also the time step for the gapfilled time series later. 

In [ ]:
ml = ps.Model(head_gap)

sm = ps.RechargeModel(
    ml,
    prec=prec,
    evap=evap,
    rfunc=ps.Exponential(),
    name="rch",
    recharge=ps.rch.FlexModel(),
)

ml.solve(report=False, tmax="2011-01-01")
ps.ArNoiseModel(ml)
ml.solve(initial=False, report=False)

We'll make a plot of all the data and the Pastas model that we generated so far. The figure below clearly shows how the simulation works quite well, but actually less good during the large gap with the minimum heads.

In [ ]:
ax = ml.plot(figsize=(10, 3))

head.plot(
    marker=".",
    color="gray",
    linestyle="None",
    label="head measured",
    markersize=3,
    zorder=1,
    ax=ax,
)

ax.grid()
ax.set_ylabel("Head (m)")

## Naive gapfilling

We'll start by filling the heads, without using the noisemodel. By default, the gapfill module will use the noisemodel if present, but we can manually turn this option off (`use_noisemodel=False`). The simulated head is used to gapfill the data, corrected by the last measurement before and the first measurement after the gap. In between, a linear interpolation is used for the correction.

By providing the `sig_level`, the lower and upper bounds for the (1-sig_level)*100%-prediction interval are computed. If use_noisemodels=False, the variance of the residuals will be used to compute the prediction interval, and will thus have a constant width.  

In [ ]:
## Some settings

start_date = "2003-01-01"
end_date = "2012-12-31"

In [ ]:
head_filled = ps.gap_fill(
    head_gap.asfreq("D"),
    ml,
    use_noisemodel=False,
    sig_level=0.05,
    tmin=start_date,
    tmax=end_date,
)

In [ ]:
ax = ml.plot(figsize=(10, 3), tmin=start_date, tmax=end_date)

head.plot(
    marker=".",
    color="gray",
    linestyle="None",
    label="head measured",
    markersize=3,
    zorder=1,
    ax=ax,
)

ax.grid()
# ax.set_xlim("2003", "2013")
ax.legend()
ax.set_ylabel("Head (m)")
head_filled["head_filled"].plot(
    label="head filled",
    markersize=3,
    zorder=2,
    ax=ax,
)
ax.fill_between(
    head_filled.index,
    head_filled["lower"],
    head_filled["upper"],
    color="lightblue",
    alpha=0.5,
)

ax.legend()

## Using the noisemodel

The methods applied above will work on any Pastas model to fill gaps in the data and obtain an uncertainty estimate. If a noisemodel is used, however, the noise model can be used to correct the simulated heads based on the last residual before and the first residual after the gap. The noisemodel quantifies how the residual behaves over time, and we can use that relationship to correct the simulation based on the expected (but unknown) residuals in the gap, conditioned on the known residuals before and after. Moreover, the prediction interval can also use this information, causing the intervals to widen further away from the last measurement.



In [ ]:
ml.plots.diagnostics();

In [ ]:
head_filled_noise = ps.gap_fill(
    head_gap.asfreq("D"),
    ml,
    use_noisemodel=True,
    sig_level=0.05,
    tmin=start_date,
    tmax=end_date,
)

In [ ]:
ax = ml.plot(figsize=(10, 3), tmin=start_date, tmax=end_date)

head.plot(
    marker=".",
    color="gray",
    linestyle="None",
    label="head measured",
    markersize=3,
    zorder=1,
    ax=ax,
)

ax.grid()
ax.legend()
ax.set_ylabel("Head (m)")
head_filled_noise["head_filled"].plot(
    label="head filled",
    markersize=3,
    zorder=2,
    ax=ax,
)


ax.fill_between(
    head_filled_noise.index,
    head_filled_noise["lower"],
    head_filled_noise["upper"],
    color="lightgray",
    alpha=0.8,
    zorder=0,
)

ax.legend()

## Performance of the gapfilled heads

In [ ]:
rsq_noise = ps.stats.metrics.rsq(
    head.loc[start_date:end_date], head_filled_noise["head_filled"]
)

rsq = ps.stats.metrics.rsq(head.loc[start_date:end_date], head_filled["head_filled"])

print("RSQ with noise model:", f"{rsq_noise:.3f}")
print("RSQ without noise model:", f"{rsq:.3f}")

## PICP

To evaluate the estimated predictino intervals we compute the prediction interval coverage probability (PICP). For the 95%-PI, the PICP value should be 0.95. Below indicates the the PI is overconfident, while above the PI is underconfident.

In [ ]:
# Align head_gap_daily with the original head index first
head_gap_daily_aligned = head_gap.asfreq("D").reindex(head.index)

# Identify gap locations (where the aligned series is NaN)
gap_locations_mask = head_gap_daily_aligned.isna()

# Get actual observations at gap locations
obs_at_gaps = head[gap_locations_mask]

# For head_filled_noise (with AR noise model)
# Get prediction intervals at gap locations
bounds_at_gaps_noise = head_filled_noise.loc[obs_at_gaps.index, ["lower", "upper"]]

# Compute PICP
picp_noise = ps.stats.metrics.picp(obs_at_gaps, bounds_at_gaps_noise)
print(f"PICP with noise model (95% intervals): {picp_noise:.3f}")

# For head_filled (without noise model)
bounds_at_gaps = head_filled.loc[obs_at_gaps.index, ["lower", "upper"]]
picp = ps.stats.metrics.picp(obs_at_gaps, bounds_at_gaps)
print(f"PICP without noise model (95% intervals): {picp:.3f}")

## Why it matters?

So why does it matter whether we gapfill and extend a head time series (preserving the original measurements where possible) instead of using the simulated heads? To illustrate this we compute the standardized groundwater index (SGI) on the original heads, the gapfilled heads, and the simulated heads. We than compare the SGI from the gapfilled and simulated heads to the SGI computed on the original heads (the true SGI).

In [ ]:
# Compute SGI for all series
sgi_head = ps.stats.sgi(head.loc[start_date:end_date].resample("ME").mean())
sgi_filled = ps.stats.sgi(head_filled["head_filled"].resample("ME").mean())
sgi_filled_noise = ps.stats.sgi(head_filled_noise["head_filled"].resample("ME").mean())
sim_monthly = ml.simulate(tmin=start_date, tmax=end_date).resample("ME").mean()
sgi_sim = ps.stats.sgi(sim_monthly)

# Compute R2 between SGI from head (truth) and each estimate
rsq_filled = ps.stats.metrics.rsq(sgi_head, sgi_filled)
rsq_filled_noise = ps.stats.metrics.rsq(sgi_head, sgi_filled_noise)
rsq_sim = ps.stats.metrics.rsq(sgi_head, sgi_sim)

# Plot all SGI series with R2 in the legend
fig, ax = plt.subplots(figsize=(10, 4))

sgi_head.plot(ax=ax, label="SGI head (truth)", linewidth=2, drawstyle="steps-mid")
sgi_filled.plot(ax=ax, label=f"SGI filled, R²={rsq_filled:.3f}", drawstyle="steps-mid")
sgi_filled_noise.plot(
    ax=ax,
    label=f"SGI filled noise, R²={rsq_filled_noise:.3f}",
    drawstyle="steps-mid",
    linestyle="--",
)
sgi_sim.plot(ax=ax, label=f"SGI simulation, R²={rsq_sim:.3f}", drawstyle="steps-mid")

ax.set_title("Standardized Groundwater Index (SGI)")
ax.set_ylabel("SGI (-)")
ax.grid(True)
ax.legend()
plt.tight_layout()

The results show that using the simulated heads is a bad idea for this example, and that the SGI computed using the gap-filled heads is much closer to the true SGI values. 